## IMPORTATION DES LIBRAIRIES 

In [1]:
# Importation des bibliothèques nécessaires
!pip install scrapy crochet
import requests 
import pandas as pd
import json
import numpy as np
import plotly.express as px
import time # Pour ajouter des pauses entre les requêtes API
import logging
import scrapy
from scrapy.crawler import CrawlerProcess
import os


In [2]:
!pip install --upgrade sqlalchemy pandas psycopg2-binary

## API COORDONNEES GPS :

In [3]:
# Liste des 35 villes cibles
villes = ["Mont-Saint-Michel","St-Malo","Bayeux","Le-Havre","Rouen","Paris","Amiens","Lille","Strasbourg",
          "Orschwiller","Colmar","Eguisheim","Besancon","Dijon","Annecy","Grenoble",
          "Lyon","Gorges-du-Verdon","Bormes-les-Mimosas","Cassis","Marseille","Aix-en-Provence",
          "Avignon","Uzes","Nimes","Aigues-Mortes","Saintes-Maries-de-la-mer","Collioure",
          "Carcassonne","Ariege","Toulouse","Montauban","Biarritz","Bayonne","La-Rochelle"]

# Initialisation des listes pour stocker les résultats
lat = []
lon = []

# URL de l'API Adresse (Gouvernement Français)
url_gouv = "https://api-adresse.data.gouv.fr/search/"

print("Récupération des coordonnées GPS via API Gouv...")

for ville in villes:
    query = ville.replace("-", " ")
    params = {'q': query, 'type': 'municipality', 'limit': 1}
    
    try:
        r = requests.get(url_gouv, params=params)
        data = r.json()
        
        # L'API Gouv renvoie les données dans un format GeoJSON (features)
        if data['features']: 
            # Attention : Dans le GeoJSON, les coordonnées sont [longitude, latitude]
            coords = data['features'][0]['geometry']['coordinates']
            lon.append(coords[0])
            lat.append(coords[1])
            print(f"Succès : {ville}")
        else:
            print(f"Non trouvé : {ville}")
            lat.append(None)
            lon.append(None)
            
    except Exception as e:
        print(f"Erreur sur la ville {ville}: {e}")
        lat.append(None)
        lon.append(None)

# Création du DataFrame de base
df_villes = pd.DataFrame({
    'Ville': villes,
    'latitude': lat,
    'longitude': lon
})

# Nettoyage des lignes sans coordonnées
df_villes = df_villes.dropna()

print("\nCoordonnées récupérées avec succès !")
print(df_villes.head())

Récupération des coordonnées GPS via API Gouv...
Succès : Mont-Saint-Michel
Succès : St-Malo
Succès : Bayeux
Succès : Le-Havre
Succès : Rouen
Succès : Paris
Succès : Amiens
Succès : Lille
Succès : Strasbourg
Succès : Orschwiller
Succès : Colmar
Succès : Eguisheim
Succès : Besancon
Succès : Dijon
Succès : Annecy
Succès : Grenoble
Succès : Lyon
Succès : Gorges-du-Verdon
Succès : Bormes-les-Mimosas
Succès : Cassis
Succès : Marseille
Succès : Aix-en-Provence
Succès : Avignon
Succès : Uzes
Succès : Nimes
Succès : Aigues-Mortes
Succès : Saintes-Maries-de-la-mer
Succès : Collioure
Succès : Carcassonne
Succès : Ariege
Succès : Toulouse
Succès : Montauban
Succès : Biarritz
Succès : Bayonne
Succès : La-Rochelle

Coordonnées récupérées avec succès !
               Ville   latitude  longitude
0  Mont-Saint-Michel  48.617623  -1.511072
1            St-Malo  48.642082  -1.988626
2             Bayeux  49.278188  -0.702334
3           Le-Havre  49.507345   0.129995
4              Rouen  49.440051   1.

API openweathermap 2.5 :

In [4]:
# clé API OpenWeatherMap
API_KEY = "55eb82d572ee384d1835c2d36b7d7198" 

def get_weather_data(row):
    url = f"https://api.openweathermap.org/data/2.5/forecast?lat={row['latitude']}&lon={row['longitude']}&appid={API_KEY}&units=metric&lang=fr"
    r = requests.get(url)
    data = r.json()
    
    temps = [item['main']['temp'] for item in data['list']]
    # Récupérer la probabilité de précipitation moyenne (pop est entre 0 et 1)
    pop_moy = np.mean([item.get('pop', 0) for item in data['list']])
    
    return pd.Series([np.mean(temps), pop_moy])

# Mise à jour du tri : priorité à la probabilité de pluie la plus faible
df_villes[['temp_moy', 'rain_pop']] = df_villes.apply(get_weather_data, axis=1)
df_villes = df_villes.sort_values(by=['rain_pop', 'temp_moy'], ascending=[True, False])
top_5_villes = df_villes.head(5)
print("Top 5 des destinations sélectionnées !")

Top 5 des destinations sélectionnées !


In [5]:
print(top_5_villes)

              Ville   latitude  longitude  temp_moy  rain_pop
31        Montauban  44.019840   1.363799  21.64600   0.03100
13            Dijon  47.331953   5.033601  19.78725   0.05550
21  Aix-en-Provence  43.541369   5.406124  23.28275   0.05850
34      La-Rochelle  46.157457  -1.170642  18.84575   0.06825
20        Marseille  43.282000   5.405000  24.69975   0.07400


In [6]:
# Enregistrement de nos dataframes en csv file :

df_villes.insert(0, 'city_id', range(1, 1 + len(df_villes)))
df_villes.to_csv('villes_gps_meteo.csv')
top_5_villes.to_csv('top_5_villes.csv')

In [7]:
top_5_villes = pd.read_csv('top_5_villes.csv', index_col=0)

In [8]:
# Afficher les 5 meilleurs destinations sur une carte :

fig = px.scatter_map(
    top_5_villes,
    lat="latitude",
    lon="longitude",
    color="temp_moy",
    size="temp_moy",
    hover_name="Ville",
    zoom=5,
    map_style="open-street-map", 
    title="Top 5 des destinations les plus ensoleillées"
)

fig.update_layout(title=dict(text="Top 5 villes pour voyager", x=0.5))
fig.show()

In [9]:
fig.write_html("carte_top_5_villes.html")

SCRAPING BOOKING 

In [10]:
!pip install selenium webdriver-manager

In [11]:
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# Configuration
chrome_options = Options()
chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-blink-features=AutomationControlled")
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36")

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=chrome_options)

# Fonction de scroll progressif pour voir + de 10 hotels
def scroll_pour_charger(driver, nb_hotels_cible=20):
    hotels_charges = 0
    tentatives = 0
    max_tentatives = 10

    while hotels_charges < nb_hotels_cible and tentatives < max_tentatives:
        # Scroll progressif vers le bas
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(random.uniform(0.8, 1.5))

        hotels_charges = len(driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]'))
        print(f"   {hotels_charges} hôtels chargés...")
        tentatives += 1

    # Scroll retour en haut (imitation d'un humain sur une page web)
    driver.execute_script("window.scrollTo(0, 0);")
    time.sleep(random.uniform(0.5, 1.0))

# On récupère les top villes
villes_list = top_5_villes['Ville'].tolist()
all_hotels = []

print(" Démarrage de l'extraction enrichie...")

try:
    for ville in villes_list:
        print(f"\n--- Ville : {ville} ---")
        url = f"https://www.booking.com/searchresults.fr.html?ss={ville}&checkin=2026-10-01&checkout=2026-10-05&group_adults=2"
        driver.get(url)

        # Pause initiale anti-détection
        time.sleep(random.uniform(2.0, 4.0))

        # Attente que les premières cartes soient chargées
        try:
            WebDriverWait(driver, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, '[data-testid="property-card"]'))
            )
        except:
            print(f" Timeout ou captcha détecté pour {ville}, on passe à la suite...")
            continue

        # Scroll progressif pour charger les 20 hôtels
        scroll_pour_charger(driver, nb_hotels_cible=20)

        # On récupère les cartes d'hôtels
        cards = driver.find_elements(By.CSS_SELECTOR, '[data-testid="property-card"]')
        print(f" {len(cards)} cartes trouvées, on prend les 20 premières.")

        for card in cards[:20]:
            try:
                # 1. NOM
                nom = card.find_element(By.CSS_SELECTOR, '[data-testid="title"]').text

                # 2. NOTE NUMÉRIQUE (ex: 8.5)
                try:
                    score = card.find_element(By.CSS_SELECTOR, '[data-testid="review-score"] div:first-child').text
                except:
                    score = "N/A"

                # 3. ÉVALUATION (ex: "Excellent")
                try:
                    eval_text = card.find_element(By.CSS_SELECTOR, '[data-testid="review-score"] div:nth-child(2) div:nth-child(1)').text
                except:
                    eval_text = "N/A"

                # 4. DESCRIPTION
                try:
                    desc = card.find_element(By.CSS_SELECTOR, '[data-testid="recommended-units"], .d6767e681c').text
                    desc = desc.replace('\n', ' ')
                except:
                    desc = "Aucune description disponible"

                # 5. URL
                try:
                    url_hotel = card.find_element(By.CSS_SELECTOR, '[data-testid="title-link"]').get_attribute('href')
                except:
                    url_hotel = "N/A"

                all_hotels.append({
                    'Ville': ville,
                    'nom_hotel': nom,
                    'note': score,
                    'évaluation': eval_text,
                    'description': desc,
                    'url_booking': url_hotel
                })
            except Exception as e:
                continue

        print(f"- {len(all_hotels)} lignes au total dans la liste.")

        # Pause entre chaque ville
        time.sleep(random.uniform(3.0, 5.0))

    # 2. CRÉATION DU DATAFRAME ET FUSION GPS
    df_hotels = pd.DataFrame(all_hotels)

    # Fusion avec les coordonnées GPS des villes
    df_final = df_hotels.merge(top_5_villes[['Ville', 'latitude', 'longitude']], on='Ville', how='left')

    # 3. EXPORT JSON ET CSV
    df_final.to_json("scraping_booking_complet.json", orient='records', indent=4, force_ascii=False)
    df_final.to_csv("scraping_booking_complet.csv", index=False, encoding='utf-8-sig')

    print("\n🏁 TERMINÉ ! Fichiers générés : 'scraping_booking_complet.json' et '.csv'")
    display(df_final.head())

finally:
    driver.quit()

 Démarrage de l'extraction enrichie...

--- Ville : Montauban ---
   25 hôtels chargés...
 25 cartes trouvées, on prend les 20 premières.
- 20 lignes au total dans la liste.

--- Ville : Dijon ---
   25 hôtels chargés...
 25 cartes trouvées, on prend les 20 premières.
- 40 lignes au total dans la liste.

--- Ville : Aix-en-Provence ---
   25 hôtels chargés...
 25 cartes trouvées, on prend les 20 premières.
- 60 lignes au total dans la liste.

--- Ville : La-Rochelle ---
   25 hôtels chargés...
 25 cartes trouvées, on prend les 20 premières.
- 80 lignes au total dans la liste.

--- Ville : Marseille ---
   25 hôtels chargés...
 25 cartes trouvées, on prend les 20 premières.
- 100 lignes au total dans la liste.

🏁 TERMINÉ ! Fichiers générés : 'scraping_booking_complet.json' et '.csv'


,Ville,nom_hotel,note,évaluation,description,url_booking,latitude,longitude
0,Montauban,L'authentique,"Avec une note de 9,3",N/A,Appartement 2 Chambres Appartement entier • 2 ...,https://www.booking.com/hotel/fr/authentique-m...,44.01984,1.363799
1,Montauban,En toute intimité dans le centre historique,"Avec une note de 9,2",N/A,Appartement 1 Chambre Appartement entier • 1 c...,https://www.booking.com/hotel/fr/en-toute-inti...,44.01984,1.363799
2,Montauban,,,N/A,,https://www.booking.com/hotel/fr/le-point-virg...,44.01984,1.363799
3,Montauban,,,N/A,,https://www.booking.com/hotel/fr/les-logis-du-...,44.01984,1.363799
4,Montauban,,,N/A,,https://www.booking.com/hotel/fr/studios-plein...,44.01984,1.363799


CREATION DE DATA LAKE  :

In [12]:
# Merger ma dataframe de la partie API (que celle des top 5 villes) et ma dataframe de la partie scraping booking :

# 1. Charger les fichiers avec les bonnes fonctions
df_villes = pd.read_csv('top_5_villes.csv')

# Utilisez read_csv car l'extension est .csv
df_hotels = pd.read_csv('scraping_booking_complet.csv') 

# 2. Fusionner les dataframes
df_merged = pd.merge(df_villes, df_hotels, on='Ville')

# 3. Enregistrement
df_merged.to_csv('villes_et_hotels.csv', index=False)

# Affichage
print(df_merged.head())

   Unnamed: 0      Ville  latitude_x  longitude_x  temp_moy  rain_pop  \
0          31  Montauban    44.01984     1.363799    21.646     0.031   
1          31  Montauban    44.01984     1.363799    21.646     0.031   
2          31  Montauban    44.01984     1.363799    21.646     0.031   
3          31  Montauban    44.01984     1.363799    21.646     0.031   
4          31  Montauban    44.01984     1.363799    21.646     0.031   

                                     nom_hotel                  note  \
0                                L'authentique  Avec une note de 9,3   
1  En toute intimité dans le centre historique  Avec une note de 9,2   
2                                          NaN                   NaN   
3                                          NaN                   NaN   
4                                          NaN                   NaN   

   évaluation                                        description  \
0         NaN  Appartement 2 Chambres Appartement entier • 2

In [13]:
#pip install boto3 #pour S3 AWS
!pip install boto3 python-dotenv

In [14]:
import os
#Vérification des variables d'environnement
access = os.getenv("AWS_ACCESS_KEY")
region = os.getenv("AWS_REGION")

print(f" Région : {region}")
print(f" Longueur Access Key : {len(access) if access else 0} caractères")

 Région : eu-north-1
 Longueur Access Key : 20 caractères


In [15]:
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError

# 1. Chargement des variables d'environnement
load_dotenv()
# L'argument override=True force le remplacement des anciennes clés en mémoire
load_dotenv(override=True) 

# Petit check de sécurité (sans afficher la clé)
print(f"Clé chargée ? {'OUI' if os.getenv('AWS_ACCESS_KEY') else 'NON'}")

# Récupération des clés
# Vérifiez que ces noms correspondent exactement à votre fichier .env
access_key = os.getenv("AWS_ACCESS_KEY")
secret_key = os.getenv("AWS_SECRET_KEY")
region = os.getenv("AWS_REGION") # Exemple: "eu-west-3" (Paris) ou "us-east-1"

# 2. Initialisation de la session AWS
# Note : Les paramètres s'écrivent impérativement en minuscules
session = boto3.Session(
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
    region_name=region
)

# Utilisation de la ressource S3
s3 = session.resource("s3")
bucket_name = 'joliesvacancesludo'
bucket = s3.Bucket(bucket_name)

# --- ÉTAPE 1 : GESTION DU BUCKET ---
try:
    print(f"Vérification du bucket : {bucket_name}...")
    
    # Correction de la condition de région
    if region == 'us-east-1' or region is None:
        s3.create_bucket(Bucket=bucket_name)
    else:
        # Pour eu-north-1 et les autres, cette configuration est requise
        s3.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f" Bucket '{bucket_name}' prêt.")

except ClientError as e:
    error_code = e.response['Error']['Code']
    if error_code in ['BucketAlreadyOwnedByYou', 'BucketAlreadyExists']:
        print(f" Le bucket '{bucket_name}' existe déjà. On continue...")
    else:
        print(f" Erreur critique : {e}")
        raise e

# --- ÉTAPE 2 : TÉLÉVERSEMENT DES FICHIERS ---
# Liste des fichiers attendus
files_to_upload = ["scraping_booking_complet.json", "villes_et_hotels.csv"]

for file_name in files_to_upload:
    if os.path.exists(file_name):
        print(f"Téléversement de '{file_name}'...")
        try:
            # On envoie le fichier local (1er arg) vers la clé S3 (2e arg)
            bucket.upload_file(file_name, file_name)
            print(f"    '{file_name}' envoyé avec succès.")
        except Exception as upload_err:
            print(f"    Échec de l'envoi pour '{file_name}' : {upload_err}")
    else:
        print(f" Fichier '{file_name}' introuvable localement. L'envoi a été sauté.")

print("\n Fin de l'opération S3 !")

Clé chargée ? OUI
Vérification du bucket : joliesvacancesludo...
 Le bucket 'joliesvacancesludo' existe déjà. On continue...
Téléversement de 'scraping_booking_complet.json'...
    'scraping_booking_complet.json' envoyé avec succès.
Téléversement de 'villes_et_hotels.csv'...
    'villes_et_hotels.csv' envoyé avec succès.

 Fin de l'opération S3 !


ETL :

In [16]:
import os, urllib.parse
import pandas as pd
import plotly.express as px
import boto3
from sqlalchemy import create_engine
from dotenv import load_dotenv

# 1. CONFIGURATION
load_dotenv(override=True)

db_user = os.getenv("RDS_USERNAME")
db_pass = os.getenv("RDS_PASSWORD")
db_host = os.getenv("RDS_HOSTNAME")
db_name = os.getenv("RDS_DBNAME")

# Sécurité si une variable manque
if not all([db_user, db_pass, db_host, db_name]):
    raise ValueError(" Une ou plusieurs variables RDS sont manquantes dans le .env")

safe_password = urllib.parse.quote_plus(db_pass)
connection_url = f"postgresql+psycopg2://{db_user}:{safe_password}@{db_host}:5432/{db_name}?sslmode=require"
engine = create_engine(connection_url)

# 2. EXTRACTION DEPUIS S3
s3_client = boto3.client(
    's3',
    aws_access_key_id=os.getenv("AWS_ACCESS_KEY"),
    aws_secret_access_key=os.getenv("AWS_SECRET_KEY"),
    region_name=os.getenv("AWS_REGION")
)

# On télécharge le fichier depuis le bucket
s3_client.download_file('joliesvacancesludo', 'villes_et_hotels.csv', 'villes_et_hotels.csv')
df_from_s3 = pd.read_csv('villes_et_hotels.csv')
print(f" Données récupérées depuis S3 : {len(df_from_s3)} lignes.")

# 3. TRANSFORMATION
def clean_hotel_data(df):
    rename_map = {
        'nom_hotel': 'name', 'ville': 'city', 'Ville': 'city',
        'latitude_y': 'latitude', 'longitude_y': 'longitude',
        'note': 'score', 'évaluation': 'voters', 'evaluation': 'voters'
    }
    df_clean = df.rename(columns=rename_map).copy()
    
    # Nettoyage de la colonne note si elle contient "Avec une note de X"
    if 'score' in df_clean.columns:
        df_clean['score'] = (
            df_clean['score']
            .astype(str)
            .str.replace(',', '.')
            .str.extract(r'([-+]?\d*\.?\d+)')
        )

    cols = ['name', 'city', 'score', 'voters', 'description', 'url_booking', 'latitude', 'longitude']
    df_clean = df_clean[[c for c in cols if c in df_clean.columns]]
    
    for col in ['score', 'latitude', 'longitude']:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean.dropna(subset=['latitude', 'longitude', 'score']).drop_duplicates()

df_hotels = clean_hotel_data(df_from_s3)

# 4. CHARGEMENT DANS RDS
try:
    with engine.begin() as conn:
        df_hotels.to_sql('hotels', con=conn, if_exists='replace', index=False)
    print(" Données synchronisées sur AWS RDS.")
except Exception as e:
    print(f" Erreur LOAD : {e}")

# 5. CARTO TOP 20
query = "SELECT * FROM hotels ORDER BY score DESC LIMIT 20"
top_20 = pd.read_sql(query, engine)

fig = px.scatter_mapbox(
    top_20, lat="latitude", lon="longitude", hover_name="name",
    color="score", size="score", color_continuous_scale="Viridis",
    zoom=4.5, mapbox_style="open-street-map",
    title=" Top 20 des Meilleurs Hôtels (Données RDS)"
)
fig.show()

 Données récupérées depuis S3 : 100 lignes.
 Données synchronisées sur AWS RDS.


C:\Users\surel\AppData\Local\Temp\ipykernel_27924\1208247244.py:77: DeprecationWarning: *scatter_mapbox* is deprecated! Use *scatter_map* instead. Learn more at: https://plotly.com/python/mapbox-to-maplibre/
  fig = px.scatter_mapbox(
